In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler 
from sklearn.metrics import cohen_kappa_score
import copy
from transformations import train_loader,val_loader,test_loader
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from tqdm import tqdm
from config import device

In [ ]:
# Load pretrained 
efficientnet = models.efficientnet_b0(weights='DEFAULT').cuda()

# Replace classifier head for binary classification
num_features = efficientnet.classifier[1].in_features
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(num_features, 1) 
)

binary_model = efficientnet.to(device)

# --- Loss and optimizer ---
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(binary_model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

# --- Early stopping parameters ---
patience = 5
best_val_loss = np.inf
counter = 0
best_model_state = None

## Model training

In [ ]:
def train_model_binary(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, patience=5):
    scaler = GradScaler()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    counter = 0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-'*10)

        # ---- TRAIN ----
        model.train()
        running_loss = 0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.float().unsqueeze(1).to(device)  # shape (B,1)

            optimizer.zero_grad()
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects / len(train_loader.dataset)
        print(f'Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}')

        # ---- VALIDATION ----
        model.eval()
        val_loss = 0
        val_corrects = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_loss += loss.item() * inputs.size(0)
                val_corrects += (preds == labels).sum().item()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_acc = val_corrects / len(val_loader.dataset)
        kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Kappa: {kappa:.4f}')

        # ---- Early stopping ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
            print("-> Best model saved!")
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_wts)
    return binary_model

In [ ]:
num_epochs = 50
patience = 5

binary_model = train_model_binary(
    binary_model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer, 
    num_epochs=num_epochs, 
    patience=patience
)

In [ ]:
import pickle
pickle.dump(binary_model, open("binary_model.pkl", "wb"))

## Model evaluatioin

In [ ]:
@torch.no_grad()
def evaluate_binary_bce(model, test_loader, device):
    model.eval()
    
    total_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []
    
    criterion = torch.nn.BCEWithLogitsLoss()   # same as training

    for images, labels in tqdm(test_loader, desc="Evaluating Test", leave=False):
        images = images.to(device)
        labels = labels.to(device).float().view(-1, 1)   # (B,) → (B,1) float

        outputs = model(images)                          # (B, 1)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * images.size(0)
        
        probs = torch.sigmoid(outputs)                   # convert logits → probability
        preds = (probs > 0.5).float()                    # default threshold

        all_probs.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

    # Metrics
    avg_loss = total_loss / len(test_loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    print("\n" + "="*70)
    print(f"{'FINAL TEST RESULTS':^70}")
    print("="*70)
    print(f"Test Loss          : {avg_loss:.4f}")
    print(f"Test Accuracy      : {accuracy*100:.2f}%")
    print(f"Test AUC           : {auc:.4f}")
    print(f"Test Quadratic Kappa : {kappa:.4f}")
    print("="*70)
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["No DR", "DR"]))

    # Bonus: find the threshold that maximizes Quadratic Weighted Kappa
    print("\nOptimizing threshold for maximum Kappa...")
    best_thr = 0.5
    best_kappa = kappa
    for thr in np.linspace(0.1, 0.9, 81):
        preds_thr = (np.array(all_probs) >= thr).astype(int)
        k = cohen_kappa_score(all_labels, preds_thr, weights='quadratic')
        if k > best_kappa:
            best_kappa = k
            best_thr = thr
    print(f"Best threshold = {best_thr:.3f} → Optimized Kappa = {best_kappa:.4f}")

    return avg_loss, accuracy, auc, kappa, best_kappa

In [ ]:
test_loss, test_acc, test_auc, test_kappa, test_kappa_opt = evaluate_binary_bce(binary_model, test_loader, device)